In [ ]:
#bloque de imports de librerias DE LOS METODOS
import pandas as pd

from pgmpy.models import BayesianNetwork #la terminal me dijo que esta obsoleto
from pgmpy.estimators import HillClimbSearch, ExhaustiveSearch, BIC, K2

#los siguientes import son para estimar los parametros de la red bayesiana, es decir, para 
#aprender las probabilidades condicionales a partir de los datos, lo cual es un paso importante despues de haber aprendido la estructura de la red
from pgmpy.estimators import MaximumLikelihoodEstimator  # esto nos servira para estimar los parametros, tal como se enseña en la documentacion
from pgmpy.models import DiscreteBayesianNetwork

#los siguientes import son para poder hacer inferencias con la red bayesiana, 
#es decir, para poder hacer consultas sobre la red y obtener probabilidades de eventos dados ciertos evidencias
from pgmpy.example_models import load_model
from pgmpy.inference import VariableElimination


#import pgmpy.estimators
#print(dir(pgmpy.estimators))     # esto sirve para poder ver que clases y funciones tiene el modulo de estimadores de pgmpy, lo cual es util para saber que opciones tenemos

#imports de librerias para graficar

import networkx as nx
import matplotlib.pyplot as plt




In [ ]:
#cargado de datos y seleccion

#creamos un array con los nombres de las variables (columnas del archivo)
#como el archivo no viene con cabeceras este array servira para definirlas
# las definiciones de cada nombre de columna estan en la documentacion y se deben añadir en el informe
col_name = ['poisonuos', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 
           'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 
           'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 
           'stalk-surface-below-ring', 'stalk-color-above-ring', 
           'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 
           'ring-type', 'spore-print-color', 'population', 'habitat']

doc_completo = pd.read_csv('./mushroom/agaricus-lepiota.data', header=None, names=col_name)


HC_col_name = ['poisonuos', 'cap-shape', 'cap-color', 'odor', 
           'stalk-shape', 'stalk-root', 'veil-type', 'ring-type']



HC_doc = doc_completo[HC_col_name].copy() # aqui tenemos 7 columnas, se reducieron para que no sea un proceso tan pesadom ademas de poder analizar mejor los resultados

ES_col_name = ['poisonuos', 'cap-color', 'odor', 'stalk-root', 'veil-type']

#el copy() es mejor que hacer un script donde se guarden solo las columnas segun el nombre, asi es mas facil
ES_doc = doc_completo[ES_col_name].copy() # asi el doc solo tendra las columnas con ese nombre, en vez de meter todas en 5 o 7 columnas



In [19]:
hc = HillClimbSearch(HC_doc) #aplicamos el metodo HC para encontrar la estructura de la red bayesiana, el resultado se guarda en la variable hc
#este metodo se basa en una busqueda local que comienza con una estructura vacia
#  y va añadiendo, eliminando o revirtiendo arcos para mejorar la puntuacion del modelo segun el criterio de BIC

#obtenemos el mejor modelo encontrado por HC utilizando el criterio de BIC para evaluar la calidad del modelo
HC_mejor_modelo = hc.estimate(scoring_method=BIC(HC_doc))
#por ejemplo un resultado podria ser: (cap-shape, cap-color) lo que indica que hay un arco dirigido desde la variable cap-shape 
# hacia la variable cap-color, lo que sugiere que el color del sombrero puede depender de su forma

#los arcos encontrados por HC representan las relaciones de dependencia entre las variables del conjunto de datos
print("Arcos encontrados: ")
print(HC_mejor_modelo.edges())


/tmp/ipykernel_26136/231762910.py:1: FutureWarning: HillClimbSearch is deprecated. Please use pgmpy.causal_discovery.HillClimbSearch instead.
  hc = HillClimbSearch(HC_doc) #aplicamos el metodo HC para encontrar la estructura de la red bayesiana, el resultado se guarda en la variable hc
  0%|          | 11/1000000 [00:00<12:48:20, 21.69it/s]

Arcos encontrados: 
[('poisonuos', 'odor'), ('poisonuos', 'cap-shape'), ('odor', 'stalk-root'), ('odor', 'ring-type'), ('odor', 'cap-color'), ('odor', 'stalk-shape'), ('stalk-shape', 'stalk-root'), ('stalk-shape', 'cap-color'), ('stalk-root', 'cap-shape'), ('ring-type', 'stalk-shape'), ('ring-type', 'stalk-root')]


In [15]:
#definir el método de puntuación con tus datos
puntuacion = K2(ES_doc)

#pasar la puntuación AL CONSTRUCTOR del objeto ExhaustiveSearch
est = ExhaustiveSearch(ES_doc, scoring_method=puntuacion)

#ejecutar el .estimate()
ES_mejor_modelo = est.estimate()

print("Arcos encontrados por Búsqueda Exhaustiva:")
print(ES_mejor_modelo.edges())

Arcos encontrados por Búsqueda Exhaustiva:
[('odor', 'cap-color'), ('odor', 'poisonuos'), ('odor', 'stalk-root'), ('poisonuos', 'cap-color'), ('poisonuos', 'stalk-root'), ('stalk-root', 'cap-color'), ('veil-type', 'poisonuos')]


In [ ]:
modelo_a_graficar = HC_mejor_modelo 

# Convertimos los arcos aprendidos en un Grafo Dirigido de NetworkX
G_hc = nx.DiGraph(modelo_a_graficar.edges())

# --- ESTILO Y DISEÑO (AQUÍ ESTÁ LO "BONITO") ---
plt.figure(figsize=(16, 10)) # Tamaño grande para que se lea bien

# Diseño: Usamos 'shell_layout' para organizar los nodos en círculos concéntricos
# Esto ordena mucho la vista cuando hay muchas variables.
pos_hc = nx.shell_layout(G_hc)

# 1. Dibujar los Nodos (Círculos)
nx.draw_networkx_nodes(G_hc, pos_hc, 
                       node_size=4000,          # Tamaño grande
                       node_color='#a1d99b',    # Verde suave (estilo hongos)
                       edgecolors='black',      # Borde negro fino
                       linewidths=1.5)

# 2. Dibujar los Arcos (Flechas)
nx.draw_networkx_edges(G_hc, pos_hc, 
                       edgelist=G_hc.edges(),
                       edge_color='#636363',    # Gris oscuro
                       arrowsize=25,            # Flechas grandes y visibles
                       arrowstyle='->',        # Estilo de flecha limpia
                       width=2.0)               # Grosor de la línea

# 3. Dibujar las Etiquetas (Nombres de variables)
nx.draw_networkx_labels(G_hc, pos_hc, 
                        font_size=12, 
                        font_family='sans-serif', 
                        font_weight='bold')

plt.title("Estructura de Red Bayesiana: Hill-Climbing Search", fontsize=20, fontweight='bold', pad=20)
plt.axis('off') # Ocultar los ejes cartesianos
plt.tight_layout() # Ajustar márgenes automáticamente
plt.show()

In [ ]:
modelo_es_a_graficar = ES_mejor_modelo

G_es = nx.DiGraph(modelo_es_a_graficar.edges())

# Si el modelo no encontró arcos (grafo vacío), avisamos para no dar error
if G_es.number_of_edges() == 0:
    print("El modelo de Búsqueda Exhaustiva no encontró ninguna conexión significativa (arcos).")
    # Dibujamos solo los nodos sueltos para que no quede en blanco
    G_es.add_nodes_from(modelo_es_a_graficar.nodes()) 

plt.figure(figsize=(12, 8)) # Un poco más pequeño porque hay menos nodos

# Diseño: Usamos 'spring_layout' que separa los nodos como si fueran imanes.
# Es ideal para ver estructuras claras en grafos pequeños.
pos_es = nx.spring_layout(G_es, k=1.0, iterations=50)

# 1. Dibujar los Nodos
nx.draw_networkx_nodes(G_es, pos_es, 
                       node_size=5000,          # Nodos aún más grandes
                       node_color='#9ecae1',    # Azul suave
                       edgecolors='black', 
                       linewidths=1.5)

# 2. Dibujar los Arcos (con K2 Score)
nx.draw_networkx_edges(G_es, pos_es, 
                       edgelist=G_es.edges(),
                       edge_color='#636363', 
                       arrowsize=30,            # Flechas extra grandes
                       arrowstyle='->',
                       arrows = True,
                       width=2.5)               # Líneas más gruesas

# 3. Dibujar las Etiquetas
nx.draw_networkx_labels(G_es, pos_es, 
                        font_size=14, 
                        font_family='sans-serif', 
                        font_weight='bold')

# --- FINALIZAR ---
plt.title("Estructura de Red Bayesiana: Exhaustive Search (K2 Score)", fontsize=20, fontweight='bold', pad=20)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Crear un objeto de dibujo con la estructura aprendida
# (Suponiendo que tu modelo se llama 'mejor_modelo_hc')
model_graph = nx.DiGraph(HC_mejor_modelo.edges())

# 2. Configurar el diseño (layout) y el tamaño de la figura
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(model_graph, k=0.5) # Ajusta 'k' para separar más los nodos

# 3. Dibujar nodos, flechas y etiquetas
nx.draw(model_graph, pos, with_labels=True, node_size=3000, 
        node_color="skyblue", font_size=10, font_weight="bold", 
        arrowsize=20, edge_color="gray")

plt.title("Estructura de la Red Bayesiana (Hill-Climbing)")
plt.show()

In [20]:
HC_modelo_final = DiscreteBayesianNetwork(HC_mejor_modelo.edges()) # tengo que ver bien que wea hace esto porque solo copie y pegue esto de la documentacion
ES_modelo_final = DiscreteBayesianNetwork(ES_mejor_modelo.edges())

#aqui hacemos la estimacion de parametros
HC_modelo_final.fit(HC_doc, estimator=MaximumLikelihoodEstimator)
ES_modelo_final.fit(ES_doc, estimator=MaximumLikelihoodEstimator)


HC_modelo_final.cpds
ES_modelo_final.cpds


# Mostrar CPDs de forma legible
#print("\n--- Parámetros de la red (CPDs) ---")
for cpd in HC_modelo_final.get_cpds():
    print(cpd)


+--------------+----------+
| poisonuos(e) | 0.517971 |
+--------------+----------+
| poisonuos(p) | 0.482029 |
+--------------+----------+
+-----------+---------------------+----------------------+
| poisonuos | poisonuos(e)        | poisonuos(p)         |
+-----------+---------------------+----------------------+
| odor(a)   | 0.09505703422053231 | 0.0                  |
+-----------+---------------------+----------------------+
| odor(c)   | 0.0                 | 0.049029622063329927 |
+-----------+---------------------+----------------------+
| odor(f)   | 0.0                 | 0.5515832482124617   |
+-----------+---------------------+----------------------+
| odor(l)   | 0.09505703422053231 | 0.0                  |
+-----------+---------------------+----------------------+
| odor(m)   | 0.0                 | 0.009193054136874362 |
+-----------+---------------------+----------------------+
| odor(n)   | 0.8098859315589354  | 0.030643513789581207 |
+-----------+---------------------

In [ ]:
#inferencias a posteriori 1 (diagnostico) (Hill-Climbing)

HC_inferencia = VariableElimination(HC_modelo_final)

#cual es la probabilidad de que sea venenoso dado que [cap-shape: convex] y el [odor: pungent] 
HC_resultado1 = HC_inferencia.query(variables=["poisonuos"], evidence={"cap-shape": "x" , "odor": "p"},)

#
HC_resultado2 = HC_inferencia.query(variables=["poisonuos"], evidence={"cap-color": "n" , "odor": "p"},)


#https://pgmpy.org/started/quickstart.html#quickstart-parameter-estimation
print(HC_resultado1)
print(HC_resultado2)

In [ ]:
#inferencias a posteriori 2 (diagnostico) (Exhaustive Search)
ES_inferencia = VariableElimination(ES_modelo_final)

#cual es la probabilidad de que sea venenoso dado que [cap-color: n] y el [odor: pungent]
ES_ressultado1 = ES_inferencia.query(variables=["poisonuos"], evidence={"cap-color": "n" , "odor": "p"},)

#cual es la probabilidad de que sea venenoso dado que [stalk-root: b] y el [veil-type: p]
ES_resultado2 = ES_inferencia.query(variables=["poisonuos"], evidence={"stalk-root": "b" , "veil-type": "p"},)  

print(ES_ressultado1)
print(ES_resultado2)


In [ ]:
# ahora hay que hacer la generacion de datos sinteticos a partir de las redes bayesianas aprendidas, 
# esto se hace para poder evaluar el rendimiento de las redes en tareas de inferencia y diagnostico, 
# ademas de poder comparar los resultados con los datos reales y ver si las redes son capaces de generalizar bien a nuevos casos. 
# Para esto se pueden usar funciones como 
# 'sample' que permiten generar muestras sintéticas a partir de la distribución aprendida por la red bayesiana.


#ejemplo para el 10% de aumento
n_original = len(HC_doc) # obtenemos el numero de muestras originales del dataset para poder calcular el 10%
n_extra_10 = int(n_original * 0.10)
n_extra_20 = int(n_original * 0.20)
n_extra_40 = int(n_original * 0.40)

df_sintetico_10 = HC_modelo_final.simulate(n_samples=n_extra_10)
df_sintetico_20 = HC_modelo_final.simulate(n_samples=n_extra_20)
df_sintetico_40 = HC_modelo_final.simulate(n_samples=n_extra_40)

#unimos los datos sinteticos generado al dataset original para crear el "Dataset Aumentado"
df_aumentado_10 = pd.concat([HC_doc, df_sintetico_10], ignore_index=True)
df_aumentado_20 = pd.concat([HC_doc, df_sintetico_20], ignore_index=True)
df_aumentado_40 = pd.concat([HC_doc, df_sintetico_40], ignore_index=True)


df_aumentado_10.to_csv('./mushroom/df_aumentado_10.csv', index=False)
df_aumentado_20.to_csv('./mushroom/df_aumentado_20.csv', index=False)
df_aumentado_40.to_csv('./mushroom/df_aumentado_40.csv', index=False)


#https://pgmpy.org/guides/simulations.html
#https://pgmpy.org/started/quickstart.html#quickstart-parameter-estimation

In [ ]:
"""
PARA HILL-CLIMBING

ahora despues de generar los datos sinteticos, tenemos que hacer lo mismo que antes, osea aprender la estrctura bayesiana 
con los metodos de Hill-Climbing y Exhaustive Search, pero esta vez con los datasets aumentados, para poder 
comparar los resultados con los datos reales y ver si las redes son capaces de generalizar bien a nuevos casos

"""
# col_name = ['poisonuos', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 
#            'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 
#            'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 
#            'stalk-surface-below-ring', 'stalk-color-above-ring', 
#            'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 
#            'ring-type', 'spore-print-color', 'population', 'habitat']

#ES_col_name = ['poisonuos', 'cap-color', 'odor', 'stalk-root', 'veil-type']


#leemos los datos aumentados, estos serian los datos completos, como antes lo teniamos "doc_completo"
df_aumentado_10 = pd.read_csv('./mushroom/df_aumentado_10.csv', header=None, names=col_name)
df_aumentado_20 = pd.read_csv('./mushroom/df_aumentado_20.csv', header=None, names=col_name)
df_aumentado_40 = pd.read_csv('./mushroom/df_aumentado_40.csv', header=None, names=col_name)


HC_doc_10 = df_aumentado_10.copy() # aqui tenemos las 23 columnas
HC_doc_20 = df_aumentado_20.copy() # aqui tenemos las 23 columnas
HC_doc_40 = df_aumentado_40.copy() # aqui tenemos las 23 columnas

#el copy() es mejor que hacer un script donde se guarden solo las columnas segun el nombre, asi es mas facil
ES_doc_10= df_aumentado_10[ES_col_name].copy()
ES_doc_20= df_aumentado_20[ES_col_name].copy()
ES_doc_40= df_aumentado_40[ES_col_name].copy()



hc_10 = HillClimbSearch(HC_doc_10) 
hc_20 = HillClimbSearch(HC_doc_20)
hc_40 = HillClimbSearch(HC_doc_40)

HC_mejor_modelo_10 = hc_10.estimate(scoring_method=BIC(HC_doc_10))
HC_mejor_modelo_20 = hc_20.estimate(scoring_method=BIC(HC_doc_20))
HC_mejor_modelo_40 = hc_40.estimate(scoring_method=BIC(HC_doc_40))

print("Arcos encontrados 10: ")
print(HC_mejor_modelo_10.edges())

print("Arcos encontrados 20: ")
print(HC_mejor_modelo_20.edges())

print("Arcos encontrados 40: ")
print(HC_mejor_modelo_40.edges())





In [ ]:
#PARA HILL-CLIMBING
#Ahora graficaremos las estructuras aprendiadas por HC

model_graph_10= nx.DiGraph(HC_mejor_modelo_10.edges())

plt.figure(figsize=(12, 8))
pos_10= nx.spring_layout(model_graph_10, k=0.5)

nx.draw(model_graph_10, pos_10, with_labels=True, node_size=3000, 
        node_color="skyblue", font_size=10, font_weight="bold", 
        arrowsize=20, edge_color="gray")


plt.title("Estructura de la Red Bayesiana (Hill-Climbing) aumentada 10%")
plt.show()





In [ ]:
model_graph_20= nx.DiGraph(HC_mejor_modelo_20.edges())
plt.figure(figsize=(12, 8))
pos_20 = nx.spring_layout(model_graph_20, k=0.5)
nx.draw(model_graph_20, pos_20, with_labels=True, node_size=3000, 
        node_color="lightcoral", font_size=10, font_weight="bold", 
        arrowsize=20, edge_color="gray")

plt.title("Estructura de la Red Bayesiana (Hill-Climbing aumentada 20%)")
plt.show()

In [ ]:
model_graph_40= nx.DiGraph(HC_mejor_modelo_40.edges())
plt.figure(figsize=(12, 8))
pos_40 = nx.spring_layout(model_graph_40, k=0.5)
nx.draw(model_graph_40, pos_40, with_labels=True, node_size=3000, 
        node_color="lightgreen", font_size=10, font_weight="bold", 
        arrowsize=20, edge_color="gray")

plt.title("Estructura de la Red Bayesiana (Hill-Climbing) aumentada 40%")
plt.show()

In [ ]:
#PARA HILL-CLIMBING
#

#ESTIMACION DE PARAMETROS PARA HC AUMENTADO 10%

HC_modelo_final_10 = DiscreteBayesianNetwork(HC_mejor_modelo_10.edges()) 

#aqui hacemos la estimacion de parametros
HC_modelo_final_10.fit(HC_doc_10, estimator=MaximumLikelihoodEstimator)
HC_modelo_final_10.cpds


In [ ]:
HC_modelo_final_20 = DiscreteBayesianNetwork(HC_mejor_modelo_20.edges())

#aqui hacemos la estimacion de parametros
HC_modelo_final_20.fit(HC_doc_20, estimator=MaximumLikelihoodEstimator)

HC_modelo_final_20.cpds


In [ ]:
HC_modelo_final_40 = DiscreteBayesianNetwork(HC_mejor_modelo_40.edges())

#aqui hacemos la estimacion de parametros
HC_modelo_final_40.fit(HC_doc_40, estimator=MaximumLikelihoodEstimator)

HC_modelo_final_40.cpds


In [ ]:
#PARA HILL-CLIMBING
#

#AHORA DESPES DE LA ESTIMACION DE PARAMETROS TENEMOS QUE HACER LAS INFERENCIAS A POSTERIORI PARA CADA MODELOS AUMENTADO ESTUDIADO CON HC

#inferencias a posteriori 1 (diagnostico) (Hill-Climbing)

HC_inferencia_10 = VariableElimination(HC_modelo_final_10)
HC_inferencia_20 = VariableElimination(HC_modelo_final_20)
HC_inferencia_40 = VariableElimination(HC_modelo_final_40)

#cual es la probabilidad de que sea venenoso dado que [cap-shape: convex] y el [odor: pungent] 
HC_resultado1_10 = HC_inferencia_10.query(variables=["poisonuos"], evidence={"cap-shape": "x" , "odor": "p"},)
HC_resultado2_10= HC_inferencia_10.query(variables=["poisonuos"], evidence={"cap-color": "n" , "odor": "p"},)

HC_resultado1_20 = HC_inferencia_20.query(variables=["poisonuos"], evidence={"cap-shape": "x" , "odor": "p"},)
HC_resultado2_20= HC_inferencia_20.query(variables=["poisonuos"], evidence={"cap-color": "n" , "odor": "p"},)

HC_resultado1_40 = HC_inferencia_40.query(variables=["poisonuos"], evidence={"cap-shape": "x" , "odor": "p"},)
HC_resultado2_40= HC_inferencia_40.query(variables=["poisonuos"], evidence={"cap-color": "b" , "odor": "s"},)


#https://pgmpy.org/started/quickstart.html#quickstart-parameter-estimation
print(HC_resultado1_10)
print(HC_resultado2_10)
print('\n')
print(HC_resultado1_20)
print(HC_resultado2_20)
print('\n')
print(HC_resultado1_40)
print(HC_resultado2_40)
